# Order Isomorphism: Divisors of 120 and Down-Sets of Join-Irreducibles

This notebook uses the preamble's `Sets().PartiallyOrdered()` category and idiomatic mathematical set comprehensions to demonstrate the order isomorphism between:
1. $\mathcal{D}(120)$: The lattice of divisors of $120 = 2^3 \cdot 3 \cdot 5$, partially ordered by divisibility ($a \le b \iff a \mid b$).
2. $\mathcal{O}(S)$: The lattice of divisor-closed subsets (down-sets / order ideals) of the poset of join-irreducible elements $S = \{2, 3, 4, 5, 8\}$, partially ordered by set inclusion ($U_1 \le U_2 \iff U_1 \subseteq U_2$).

### Theoretical Background: Birkhoff's Representation Theorem
By **Birkhoff's Representation Theorem for Finite Distributive Lattices**, any finite distributive lattice $L$ is naturally order-isomorphic to the lattice of down-sets $\mathcal{O}(J(L))$ of its poset of join-irreducible elements $J(L)$:
$$
L \cong \mathcal{O}(J(L))
$$
For $L = \mathcal{D}(120)$:
- The join-irreducible elements are prime powers $p^k > 1$ dividing $120$:
  $$J(\mathcal{D}(120)) = \{2^1, 2^2, 2^3, 3^1, 5^1\} = \{2, 4, 8, 3, 5\} = S$$
- The canonical order-preserving bijection $\Phi: \mathcal{D}(120) \xrightarrow{\cong} \mathcal{O}(S)$ is the set comprehension:
  $$\Phi(d) = \{s \in S \mid d \equiv 0 \pmod s\}$$
- The inverse map is:
  $$\Psi(U) = \operatorname{lcm}(U \cup \{1\})$$

In [ ]:
from sage.all import *
from sage.combinat.posets.posets import Poset
from dzack_research.preamble.categories.sets.owned_sets import Sets, placement_of

print('Owned Poset Category:', Sets().PartiallyOrdered().Finite())

## 1. Construct $\mathcal{D}(120)$ (Divisors Poset)

In [ ]:
# Idiomatic set comprehension for divisors of 120
divs_120 = {d for d in 120.divisors()}
P_div = Poset((list(divs_120), lambda a, b: b % a == 0))

print(f'Divisors count: {len(divs_120)}')
print(f'Divisors: {sorted(list(divs_120))}')
print(f'Category placement: {placement_of(P_div)}')

## 2. Construct $\mathcal{O}(S)$ (Divisor-Closed Subsets of $S = \{2, 3, 4, 5, 8\}$)

In [ ]:
# Join-irreducible elements
S = {2, 3, 4, 5, 8}

# Idiomatic set comprehension for down-sets / divisor-closed subsets in S
closed_subsets = {U for U in S.subsets() if all(y in U for x in U for y in S if x % y == 0)}

P_closed = Poset((list(closed_subsets), lambda U1, U2: U1.issubset(U2)))

print(f'Divisor-closed subsets count: {len(closed_subsets)}')
print(f'Category placement: {placement_of(P_closed)}')

## 3. Order Isomorphism and Explicit Bijection

In [ ]:
# Verify isomorphism
assert P_div.is_isomorphic(P_closed), 'Posets must be order-isomorphic!'
print('✓ P_div is order-isomorphic to P_closed')

# Forward and inverse morphisms using mathematical set comprehensions
phi = lambda d: {s in S | d % s == 0}
psi = lambda U: Integer(lcm(list(U) + [1]))

print('\nExplicit Bijection Table:')
print('=' * 50)
print(f"{'Divisor d':<12} | {'Subset phi(d) in S':<25} | {'psi(phi(d))':<10}")
print('-' * 50)
for d in sorted(list(divs_120)):
    U = phi(d)
    d_rec = psi(U)
    assert d == d_rec, f'Roundtrip failed for {d}'
    subset_str = str(sorted(list(U)))
    print(f"{d:<12} | {subset_str:<25} | {d_rec:<10}")
print('=' * 50)
print('✓ All roundtrips psi(phi(d)) == d verified!')

## 4. Verification of Order Preservation
We verify that $d_1 \mid d_2 \iff \Phi(d_1) \subseteq \Phi(d_2)$ for all $16 \times 16 = 256$ pairs.

In [ ]:
order_preserving = all(
    (d2 % d1 == 0) == all(s in phi(d2) for s in phi(d1))
    for d1 in divs_120
    for d2 in divs_120
)

assert order_preserving, 'Order preservation failed!'
print('✓ phi strictly preserves and reflects the partial order:')
print('  d1 | d2  <===>  phi(d1) subseteq phi(d2)  for all d1, d2 in Div(120)')

## 5. Join-Irreducible Elements & Poset Factorization
The join-irreducible elements of $\mathcal{D}(120)$ are identified via set comprehension of single-lower-cover elements.

In [ ]:
# Find join-irreducibles using set comprehension
join_irreducibles = {x for x in P_div if len(P_div.lower_covers(x)) == 1}
print(f'Join-irreducibles of P_div: {sorted(list(join_irreducibles))}')
assert join_irreducibles == S, 'Join-irreducibles must match S = {2, 3, 4, 5, 8}'
print('✓ J(Div(120)) = {2, 3, 4, 5, 8} exactly matches S!')